# Lab 26-05 · Schedules, Respins and When to Tape Out

Companion to chapter 26-05. Write down each prediction before running the cell.

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import risk as r

## Part 1 · Monte Carlo schedule

*Predict:* higher or lower than the normal approximation's 0.826 chance of tape-out by week 20?

In [2]:
A = [(10, 14, 24), (1, 2, 4)]     # verification closure, final regression
B = [(8, 11, 18), (4, 5, 9)]      # synthesis and place and route, signoff
pa, pb = r.path_stats(A), r.path_stats(B)
print(f"normal approximation: P(both by 20) = {r.p_all_by(20, [pa, pb]):.3f}, 90% date {r.date_for(0.9, [pa, pb]):.1f} weeks")
sim = r.simulate_tapeout([A, B])
print(f"Monte Carlo: P(both by 20) = {(sim <= 20).mean():.3f}, 90% date {np.quantile(sim, 0.9):.1f} weeks, mean {sim.mean():.2f}")
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.hist(sim, bins=80, density=True)
ax.axvline(20, color="k", ls=":"); ax.set_xlabel("tape-out week"); ax.set_ylabel("density")

normal approximation: P(both by 20) = 0.826, 90% date 20.7 weeks
Monte Carlo: P(both by 20) = 0.768, 90% date 21.2 weeks, mean 18.48


Text(0, 0.5, 'density')

## Part 2 · Learning between spins

*Predict:* the expected number of tape-outs if A0's success probability rises from 0.25 to 0.50.

In [3]:
for q1 in (0.25, 0.50):
    print(f"q1 = {q1:.2f}: E[N] = {r.expected_spins([q1, 0.70, 0.95, 1.0]):.3f}")
q1s = np.linspace(0, 1, 101)
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(q1s, [r.expected_spins([q, 0.70, 0.95, 1.0]) for q in q1s])
ax.set_xlabel("first-spin success probability"); ax.set_ylabel("expected tape-outs")

q1 = 0.25: E[N] = 1.986
q1 = 0.50: E[N] = 1.657


Text(0, 0.5, 'expected tape-outs')

## Part 3 · The cost of stopping

*Predict:* the shape of the expected cost between 0 and 30 weeks, and its value at 24 weeks.

In [4]:
t = np.linspace(0, 30, 301)
f0 = [r.total_cost(x, 40, 4.0, 0.25, 1.5, 40) for x in t]
f1 = [r.total_cost(x, 40, 2.5, 0.25, 1.5, 40, fixed=3.0) for x in t]
print(f"cost at 24 weeks: ${r.total_cost(24, 40, 4.0, 0.25, 1.5, 40):.1f} M")
i_max = int(np.argmax(f0[:150])); i_min = int(np.argmin(f0))
print(f"ridge near {t[i_max]:.1f} weeks (${f0[i_max]:.1f} M); minimum near {t[i_min]:.1f} weeks (${f0[i_min]:.1f} M)")
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(t, f0, label="no emulation"); ax.plot(t, f1, label="emulation ($3 M)")
ax.set_xlabel("weeks of final verification"); ax.set_ylabel("expected cost ($M)"); ax.legend()

cost at 24 weeks: $37.0 M
ridge near 4.8 weeks ($45.2 M); minimum near 16.1 weeks ($30.7 M)


## Part 4 · What emulation must deliver

*Predict:* how far must emulation shorten the mean discovery time (from 4 weeks) to repay $3 M?

In [5]:
from scipy import optimize
def best(tau):
    ts, _ = r.optimal_tapeout(40, tau, 0.25, 1.5, 40)
    return r.total_cost(ts, 40, tau, 0.25, 1.5, 40)
base = best(4.0)
tau_be = optimize.brentq(lambda x: best(x) + 3.0 - base, 0.5, 3.99)
print(f"emulation pays if tau < {tau_be:.2f} weeks ({100 * (1 - tau_be / 4):.0f}% faster discovery)")
taus = np.linspace(1, 6, 101)
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(taus, [best(x) for x in taus]); ax.axhline(base - 3, ls="--", color="gray")
ax.set_xlabel("mean discovery time (weeks)"); ax.set_ylabel("minimum expected cost ($M)")

emulation pays if tau < 3.51 weeks (12% faster discovery)


Text(0, 0.5, 'minimum expected cost ($M)')